In [1]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import os

dir = '/content/drive/MyDrive/tern_project/Chicks/terns-project-chick/TrackingTerns'

os.chdir(dir)

In [3]:
def isFileFromDates(path, dates):
    return any(date in path for date in dates)

def get_dir_path_with_parent(directory_path):
  # Get the parent directory and the base name of the directory
  parent_dir, dir_name = os.path.split(os.path.normpath(directory_path))

  # Get the parent directory name and concatenate it with the directory name
  return os.path.join("/", os.path.basename(parent_dir), dir_name)[1:]

### Read images director path and date to detect from configuration file

In [4]:
import configparser

config = configparser.ConfigParser()
# Read the config file
config.read('./track_scan_runner.ini', encoding="utf8")
# List of dates to run Yolo on
dates = config.get('General', 'dates').split(',')
# Yolo results directory path
yolo_result_dir = config.get('General', 'yolo_result_dir')
# Tracker result output directory path
tracker_result_dir = config.get('General', 'tracker_result_dir')

In [9]:
print ("dates:", dates)
print("yolo_result_dir:", yolo_result_dir)
print("tracker_result_dir:", tracker_result_dir)


dates: ['2025_06_12']
yolo_result_dir: ./../YoloDetector/YoloResults/2025
tracker_result_dir: ./TrackerResults/OneScanTracks/2025


In [10]:
import glob

# Get all directories two levels under images_dir_path
two_level_dirs = glob.glob(os.path.join(yolo_result_dir, '*', '*'))

# Filter out only directories
two_level_dirs = [get_dir_path_with_parent(path) for path in two_level_dirs \
                  if isFileFromDates(path, dates)]
two_level_dirs

['atlitcam181.stream_2025_06_12_09_01_50/tour0',
 'atlitcam181.stream_2025_06_12_09_01_50/tour1',
 'atlitcam181.stream_2025_06_12_14_01_50/tour0',
 'atlitcam181.stream_2025_06_12_14_01_50/tour1',
 'atlitcam191.stream_2025_06_12_08_59_50/tour0',
 'atlitcam191.stream_2025_06_12_08_59_50/tour1',
 'atlitcam191.stream_2025_06_12_13_59_50/tour0',
 'atlitcam191.stream_2025_06_12_13_59_50/tour1']

In [11]:
two_level_dirs = two_level_dirs[6:]
two_level_dirs

['atlitcam191.stream_2025_06_12_13_59_50/tour0',
 'atlitcam191.stream_2025_06_12_13_59_50/tour1']

### Track terns across multiple scans

In [12]:
import subprocess

for two_level_dir in two_level_dirs:

    print(f'Tracking flags in: {two_level_dir}')

    yolo_result_dir = f"-y {two_level_dir}"  # Yolo result directory realtive path passed as argument
    command = f"python ./track_terns_on_movie_chick.py {yolo_result_dir}"
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    # Check if the command ran successfully
    if result.returncode == 0:
        print(f'Successfully ran command for {two_level_dir}')
    else:
        print(f'Error occurred while processing {two_level_dir}')
        print(f'STDERR: {result.stderr}')

Tracking flags in: atlitcam191.stream_2025_06_12_13_59_50/tour0
Successfully ran command for atlitcam191.stream_2025_06_12_13_59_50/tour0
Tracking flags in: atlitcam191.stream_2025_06_12_13_59_50/tour1
Successfully ran command for atlitcam191.stream_2025_06_12_13_59_50/tour1


Read JSONs data of breeding tracks